In [ ]:
!pip install pymupdf pandas openpyxl rapidfuzz -q

import fitz
import pandas as pd
import re
from rapidfuzz import fuzz
from google.colab import files

uploaded = files.upload()

Saving teacher_guide.pdf to teacher_guide (1).pdf
Saving video_extracted_skills.csv to video_extracted_skills.csv


In [ ]:
pdf_path = [f for f in uploaded.keys() if f.lower().endswith(".pdf")][0]
csv_path = [f for f in uploaded.keys() if f.lower().endswith(".csv")][0]

df_lessons = pd.read_csv(csv_path)
df_lessons.head()

,file_name,unit_number,unit_title,lesson_number,lesson_title,skill,concept,learning_objective,difficulty_level,evidence_from_transcript
0,NA - الدرس 10 ｜ الوحدة4 ｜ وحدات قياس الطول ...,4,وحدات قياس الطول,10.0,التعرف على المتر والسنتيمتر والعلاقة بينهما,"التذكر, الملاحظة, استخدام الرموز والاختصارات ل...","المسطرة المترية, وحدة قياس المتر, وحدة قياس ال...","أن يتذكر الطالب مفهوم المسطرة المترية وقياسها,...",Easy,"ألاحظ المصطر المترية وأتذكر, نقول مصطر مترية ل..."
1,NA - -2021- الرياضيات ｜ الوحدة 1 ｜ التقريب ｜ ا...,1,التقريب,4.0,التقريب,مقارنة الأعداد ضمن 9999,مقارنة الأعداد، القيمة المنزلية (آحاد، عشرات، ...,أن يقارن الطالب بين عددين مكونين من أربع منازل...,Easy,السؤال يقول: أيهما أكبر، عدد المواليد الذكور أ...
2,NA - -2021- الرياضيات ｜ الوحدة 1 ｜ التقريب ｜ ا...,1,التقريب,4.0,التقريب,التقدير في المواقف الحياتية,التقريب كمفهوم للتقدير، عدم الحاجة للدقة المطل...,أن يفهم الطالب الحاجة إلى التقدير والتقريب في ...,Easy,ما عدد أوراق الأشجار التي في الصورة؟ ... هل نس...
3,NA - -2021- الرياضيات ｜ الوحدة 1 ｜ التقريب ｜ ا...,1,التقريب,4.0,التقريب,تحديد الأقرب في سياقات حياتية,مفهوم القرب والبعد كمدخل للتقريب.,أن يربط الطالب مفهوم القرب والبعد في الحياة ال...,Easy,إلى أي محطة سأذهب؟ ... المحطة الثانية؛ لأنها أ...
4,NA - -2021- الرياضيات ｜ الوحدة 1 ｜ التقريب ｜ ا...,1,التقريب,4.0,التقريب,تقريب الأعداد ضمن 9999,التقريب لأقرب 10، التقريب لأقرب 100، التقريب ل...,أن يقرب الطالب أعداداً ضمن 9999 إلى أقرب 10 و ...,Medium,يتوقع منا بعد الانتهاء من هذا الدرس أن نكون قا...


In [ ]:
lesson_col = "lesson_title"  # غيّريه حسب اسم العمود عندك
lessons = df_lessons[lesson_col].dropna().astype(str).tolist()

In [ ]:
def normalize_arabic(text):
    text = str(text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'[ًٌٍَُِّْـ]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

doc = fitz.open(pdf_path)

pages = []
for i, page in enumerate(doc):
    text = page.get_text("text")
    pages.append({
        "page_number": i + 1,
        "text": text,
        "norm_text": normalize_arabic(text)
    })

len(pages)

104

In [ ]:
patterns = {
    "unit_objectives": [
        "الاهداف",
        "مستويات الاهداف",
        "معرفه",
        "تطبيق",
        "استدلال"
    ],
    "conceptual_errors": [
        "الاخطاء المفاهيميه",
        "الخطا المفاهيمي",
        "اجراءات مقترحه للعلاج"
    ],
    "lesson_implementation": [
        "اليات تنفيذ الدرس",
        "اسم الدرس",
        "مرحله الاستعداد",
        "الاهداف التعليميه",
        "المهارات",
        "الخبرات السابقه",
        "المفاهيم الخاطئه المتوقعه",
        "العلاج",
        "اصول التدريس"
    ],
    "assessment": [
        "سلم تقدير عددي",
        "قائمه الرصد",
        "النشاط الكاشف",
        "مؤشرات الاداء"
    ]
}

patterns = {
    k: [normalize_arabic(x) for x in v]
    for k, v in patterns.items()
}

In [ ]:
results = []

for lesson in lessons:
    norm_lesson = normalize_arabic(lesson)

    for page in pages:
        text = page["norm_text"]

        lesson_score = fuzz.partial_ratio(norm_lesson, text)

        matched_sections = []

        for section_name, section_patterns in patterns.items():
            count = 0
            matched_words = []

            for p in section_patterns:
                if p in text:
                    count += 1
                    matched_words.append(p)

            if count > 0:
                matched_sections.append({
                    "section": section_name,
                    "matched_count": count,
                    "matched_words": matched_words
                })

        if lesson_score >= 70 or matched_sections:
            for sec in matched_sections:
                results.append({
                    "lesson_title": lesson,
                    "page_number": page["page_number"],
                    "lesson_match_score": lesson_score,
                    "section_type": sec["section"],
                    "matched_count": sec["matched_count"],
                    "matched_keywords": " | ".join(sec["matched_words"]),
                    "page_preview": page["text"][:500]
                })

results_df = pd.DataFrame(results)
results_df.head()

,lesson_title,page_number,lesson_match_score,section_type,matched_count,matched_keywords,page_preview
0,التعرف على المتر والسنتيمتر والعلاقة بينهما,3,55.813953,unit_objectives,1,معرفه,تــقــديــم\nيتصـف الҙإ صـلاح التربـوي ب�أنـه ...
1,التعرف على المتر والسنتيمتر والعلاقة بينهما,4,55.813953,unit_objectives,1,تطبيق,مــقــدمــة\nيُعدّ دليل المعلم متمّماً للصورة ...
2,التعرف على المتر والسنتيمتر والعلاقة بينهما,4,55.813953,lesson_implementation,1,المهارات,مــقــدمــة\nيُعدّ دليل المعلم متمّماً للصورة ...
3,التعرف على المتر والسنتيمتر والعلاقة بينهما,6,58.139535,unit_objectives,2,معرفه | تطبيق,2\n:نظريّات التعلّم\n الاتّجاه التقليدي في الف...
4,التعرف على المتر والسنتيمتر والعلاقة بينهما,7,55.813953,unit_objectives,1,معرفه,3\n:) الاتّجاه الحديث في التربية (النظرية البن...


In [ ]:
if not results_df.empty:
    results_df = results_df.sort_values(
        by=["lesson_title", "page_number", "matched_count", "lesson_match_score"],
        ascending=[True, True, False, False]
    )

results_df.to_csv("teacher_guide_detected_pages.csv", index=False, encoding="utf-8-sig")
files.download("teacher_guide_detected_pages.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
summary_df = results_df.groupby(["lesson_title", "section_type"])["page_number"] \
    .apply(lambda x: sorted(set(x))) \
    .reset_index()

summary_df.to_csv("teacher_guide_pages_summary.csv", index=False, encoding="utf-8-sig")
files.download("teacher_guide_pages_summary.csv")

summary_df.head(20)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,lesson_title,section_type,page_number
0,الأعداد ضمن 9999,assessment,"[37, 38, 44, 45, 49, 50, 51, 56, 57, 58, 61, 6..."
1,الأعداد ضمن 9999,conceptual_errors,"[34, 41, 54, 59, 66, 73, 79, 86, 92]"
2,الأعداد ضمن 9999,lesson_implementation,"[4, 8, 19, 20, 21, 27, 28, 30, 35, 36, 42, 43,..."
3,الأعداد ضمن 9999,unit_objectives,"[3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 17, 18, 19,..."
4,الأعداد ضمن 99999,assessment,"[37, 38, 44, 45, 49, 50, 51, 56, 57, 58, 61, 6..."
5,الأعداد ضمن 99999,conceptual_errors,"[34, 41, 54, 59, 66, 73, 79, 86, 92]"
6,الأعداد ضمن 99999,lesson_implementation,"[4, 8, 19, 20, 21, 27, 28, 30, 35, 36, 42, 43,..."
7,الأعداد ضمن 99999,unit_objectives,"[3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 17, 18, 19,..."
8,التعرف على المتر والسنتيمتر والعلاقة بينهما,assessment,"[37, 38, 44, 45, 49, 50, 51, 56, 57, 58, 61, 6..."
9,التعرف على المتر والسنتيمتر والعلاقة بينهما,conceptual_errors,"[34, 41, 54, 59, 66, 73, 79, 86, 92]"


In [ ]:
!pip install pandas rapidfuzz -q

import pandas as pd
import re
from rapidfuzz import fuzz
from google.colab import files

uploaded = files.upload()

Saving teacher_guide_detected_pages.csv to teacher_guide_detected_pages (1).csv
Saving teacher_guide_pages_summary.csv to teacher_guide_pages_summary (1).csv
Saving video_extracted_skills.csv to video_extracted_skills (1).csv


In [ ]:
guide_pages = pd.read_csv("teacher_guide_detected_pages (1).csv")
video_skills = pd.read_csv("video_extracted_skills (1).csv")

guide_pages.head(), video_skills.head()

(       lesson_title  page_number  lesson_match_score     section_type  \
 0  الأعداد ضمن 9999            3               56.25  unit_objectives   
 1  الأعداد ضمن 9999            3               56.25  unit_objectives   
 2  الأعداد ضمن 9999            3               56.25  unit_objectives   
 3  الأعداد ضمن 9999            3               56.25  unit_objectives   
 4  الأعداد ضمن 9999            3               56.25  unit_objectives   
 
    matched_count matched_keywords  \
 0              1            معرفه   
 1              1            معرفه   
 2              1            معرفه   
 3              1            معرفه   
 4              1            معرفه   
 
                                         page_preview  
 0  تــقــديــم\nيتصـف الҙإ صـلاح التربـوي ب�أنـه ...  
 1  تــقــديــم\nيتصـف الҙإ صـلاح التربـوي ب�أنـه ...  
 2  تــقــديــم\nيتصـف الҙإ صـلاح التربـوي ب�أنـه ...  
 3  تــقــديــم\nيتصـف الҙإ صـلاح التربـوي ب�أنـه ...  
 4  تــقــديــم\nيتصـف الҙإ صـلاح التربـوي ب

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def normalize_ar(text):
    text = str(text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'[ًٌٍَُِّْـ]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_skills_from_text(text):
    text = clean_text(text)

    skill_patterns = [
        r"أن\s+يقرأ\s+الطالب[^.،\n]*",
        r"أن\s+يكتب\s+الطالب[^.،\n]*",
        r"أن\s+يمثل\s+الطالب[^.،\n]*",
        r"أن\s+يتعرف\s+الطالب[^.،\n]*",
        r"أن\s+يذكر\s+الطالب[^.،\n]*",
        r"أن\s+يقارن\s+الطالب[^.،\n]*",
        r"أن\s+يرتب\s+الطالب[^.،\n]*",
        r"أن\s+يقرب\s+الطالب[^.،\n]*",
        r"قراءة\s+[^.،\n]*",
        r"كتابة\s+[^.،\n]*",
        r"تمثيل\s+[^.،\n]*"
    ]

    skills = []
    for pattern in skill_patterns:
        matches = re.findall(pattern, text)
        skills.extend(matches)

    skills = [clean_text(s) for s in skills if len(clean_text(s)) > 10]
    skills = list(dict.fromkeys(skills))

    return skills

In [ ]:
guide_skills_rows = []

for _, row in guide_pages.iterrows():
    lesson = row["lesson_title"]
    page = row["page_number"]
    text = row.get("page_preview", "")

    skills = extract_skills_from_text(text)

    for skill in skills:
        guide_skills_rows.append({
            "lesson_title": lesson,
            "page_number": page,
            "guide_skill": skill,
            "guide_skill_norm": normalize_ar(skill)
        })

guide_skills_df = pd.DataFrame(guide_skills_rows)

guide_skills_df.to_csv("guide_extracted_skills.csv", index=False, encoding="utf-8-sig")
files.download("guide_extracted_skills.csv")

guide_skills_df.head()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,lesson_title,page_number,guide_skill,guide_skill_norm
0,الأعداد ضمن 9999,34,قراءة العدد بالطُّرق المختلفة āإذا كان āأحد āأ...,قراءه العدد بالطرق المختلفه āاذا كان āاحد āارق...
1,الأعداد ضمن 9999,34,تمثيل العدد على لوحة المنازل,تمثيل العدد علي لوحه المنازل
2,الأعداد ضمن 9999,34,تمثيل على المِعداد قد يخطئ الطا,تمثيل علي المعداد قد يخطئ الطا
3,الأعداد ضمن 9999,34,قراءة العدد بالطُّرق المختلفة āإذا كان āأحد āأ...,قراءه العدد بالطرق المختلفه āاذا كان āاحد āارق...
4,الأعداد ضمن 9999,34,تمثيل العدد على لوحة المنازل,تمثيل العدد علي لوحه المنازل


In [ ]:
!pip install pandas rapidfuzz -q

import pandas as pd
import re
from rapidfuzz import fuzz
from google.colab import files

guide_df = pd.read_csv("guide_extracted_skills.csv")
video_df = pd.read_csv("video_extracted_skills (1).csv")

guide_lesson_col = "lesson_title"
guide_skill_col  = "guide_skill"

video_lesson_col = "lesson_title"
video_skill_col  = "skill"

def normalize_ar(text):
    text = str(text)
    text = re.sub(r'[إأآاٱ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'[ًٌٍَُِّْـ]', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

guide_df["lesson_norm"] = guide_df[guide_lesson_col].apply(normalize_ar)
guide_df["skill_norm"] = guide_df[guide_skill_col].apply(normalize_ar)

video_df["lesson_norm"] = video_df[video_lesson_col].apply(normalize_ar)
video_df["skill_norm"] = video_df[video_skill_col].apply(normalize_ar)

comparison_rows = []
threshold = 75

for lesson_norm in guide_df["lesson_norm"].dropna().unique():

    guide_lesson = guide_df[guide_df["lesson_norm"] == lesson_norm]
    video_lesson = video_df[video_df["lesson_norm"] == lesson_norm]

    lesson_title = guide_lesson[guide_lesson_col].iloc[0]

    for _, g in guide_lesson.iterrows():

        best_score = 0
        best_video_skill = None

        for _, v in video_lesson.iterrows():
            score = fuzz.token_set_ratio(g["skill_norm"], v["skill_norm"])

            if score > best_score:
                best_score = score
                best_video_skill = v[video_skill_col]

        comparison_rows.append({
            "lesson_title": lesson_title,
            "guide_skill": g[guide_skill_col],
            "best_video_skill_match": best_video_skill,
            "similarity_score": best_score,
            "matched": best_score >= threshold
        })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df.to_csv("skills_comparison_results.csv", index=False, encoding="utf-8-sig")
files.download("skills_comparison_results.csv")

comparison_df.head()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,lesson_title,guide_skill,best_video_skill_match,similarity_score,matched
0,الأعداد ضمن 9999,قراءة العدد بالطُّرق المختلفة āإذا كان āأحد āأ...,كتابة الأعداد المكونة من أربع منازل بالرموز من...,46.052632,False
1,الأعداد ضمن 9999,تمثيل العدد على لوحة المنازل,تمثيل الأعداد ضمن 9999 على لوحة المنازل,88.000000,True
2,الأعداد ضمن 9999,تمثيل على المِعداد قد يخطئ الطا,تمثيل الأعداد ضمن 9999 على المعداد,72.340426,False
3,الأعداد ضمن 9999,قراءة العدد بالطُّرق المختلفة āإذا كان āأحد āأ...,كتابة الأعداد المكونة من أربع منازل بالرموز من...,46.052632,False
4,الأعداد ضمن 9999,تمثيل العدد على لوحة المنازل,تمثيل الأعداد ضمن 9999 على لوحة المنازل,88.000000,True


In [ ]:
!pip install pandas rapidfuzz -q

import pandas as pd
import re
from rapidfuzz import fuzz
from google.colab import files

guide_df = pd.read_csv("guide_extracted_skills.csv")
video_df = pd.read_csv("video_extracted_skills (1).csv")

guide_lesson_col = "lesson_title"
guide_skill_col  = "guide_skill"

video_lesson_col = "lesson_title"
video_skill_col  = "skill"

In [ ]:
def normalize_ar(text):
    text = str(text)
    text = re.sub(r'[إأآاٱ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'[ًٌٍَُِّْـ]', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def split_skills(text):
    text = str(text)
    parts = re.split(r'،|,|؛|;|\n| و(?=أن )| و(?=\w)', text)
    parts = [p.strip() for p in parts if len(p.strip()) >= 8]
    return parts

def clean_skill_text(text):
    text = str(text).strip()
    text = re.sub(r'^(أن\s+)?', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
guide_rows = []

for _, row in guide_df.iterrows():
    lesson = row[guide_lesson_col]
    skill_text = row[guide_skill_col]

    for skill in split_skills(skill_text):
        skill = clean_skill_text(skill)
        guide_rows.append({
            "lesson_title": lesson,
            "skill": skill,
            "skill_norm": normalize_ar(skill)
        })

guide_clean = pd.DataFrame(guide_rows)

guide_clean = guide_clean.drop_duplicates(
    subset=["lesson_title", "skill_norm"]
).reset_index(drop=True)

guide_clean.to_csv("guide_skills_cleaned.csv", index=False, encoding="utf-8-sig")
files.download("guide_skills_cleaned.csv")

guide_clean.shape

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

(1224, 3)

In [ ]:
video_rows = []

for _, row in video_df.iterrows():
    lesson = row[video_lesson_col]
    skill_text = row[video_skill_col]

    for skill in split_skills(skill_text):
        skill = clean_skill_text(skill)
        video_rows.append({
            "lesson_title": lesson,
            "skill": skill,
            "skill_norm": normalize_ar(skill)
        })

video_clean = pd.DataFrame(video_rows)

video_clean = video_clean.drop_duplicates(
    subset=["lesson_title", "skill_norm"]
).reset_index(drop=True)

video_clean.to_csv("video_skills_cleaned.csv", index=False, encoding="utf-8-sig")
files.download("video_skills_cleaned.csv")

video_clean.shape

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

(670, 3)

In [ ]:
comparison_rows = []
threshold = 75

for lesson in guide_clean["lesson_title"].dropna().unique():

    guide_lesson = guide_clean[guide_clean["lesson_title"] == lesson]
    video_lesson = video_clean[video_clean["lesson_title"].apply(normalize_ar) == normalize_ar(lesson)]

    for _, g in guide_lesson.iterrows():

        best_score = 0
        best_video_skill = None

        for _, v in video_lesson.iterrows():
            score = fuzz.token_set_ratio(g["skill_norm"], v["skill_norm"])

            if score > best_score:
                best_score = score
                best_video_skill = v["skill"]

        comparison_rows.append({
            "lesson_title": lesson,
            "guide_skill": g["skill"],
            "best_video_skill_match": best_video_skill,
            "similarity_score": best_score,
            "matched": best_score >= threshold
        })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df.to_csv("skills_comparison_after_cleaning.csv", index=False, encoding="utf-8-sig")
files.download("skills_comparison_after_cleaning.csv")

comparison_df.head()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,lesson_title,guide_skill,best_video_skill_match,similarity_score,matched
0,الأعداد ضمن 9999,قراءة العدد بالطُّرق المختلفة āإذا كان āأحد āأ...,كتابة الأعداد المكونة من آلاف صحيحة بالرموز من...,51.282051,False
1,الأعداد ضمن 9999,حيث يُهملُ الطالب الصّفر,الأجسام الحسابية,48.648649,False
2,الأعداد ضمن 9999,يقرāأ العدد بدونه,إيجاد العدد السابق,57.142857,False
3,الأعداد ضمن 9999,تمثيل العدد على لوحة المنازل,تمثيل الأعداد ضمن 9999 على لوحة المنازل,88.000000,True
4,الأعداد ضمن 9999,تمثيل على المِعداد قد يخطئ الطا,تمثيل الأعداد ضمن 9999 على المعداد,72.340426,False


In [ ]:
lesson_accuracy = comparison_df.groupby("lesson_title").agg(
    guide_skills_count=("guide_skill", "count"),
    matched_skills_count=("matched", "sum"),
    average_similarity=("similarity_score", "mean")
).reset_index()

lesson_accuracy["coverage_accuracy"] = (
    lesson_accuracy["matched_skills_count"] / lesson_accuracy["guide_skills_count"]
) * 100

lesson_accuracy.to_csv("lesson_skill_accuracy_after_cleaning.csv", index=False, encoding="utf-8-sig")
files.download("lesson_skill_accuracy_after_cleaning.csv")

lesson_accuracy

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,lesson_title,guide_skills_count,matched_skills_count,average_similarity,coverage_accuracy
0,الأعداد ضمن 9999,24,3,59.379990,12.500000
1,الأعداد ضمن 99999,24,5,60.547786,20.833333
2,التعرف على المتر والسنتيمتر والعلاقة بينهما,24,0,37.012774,0.000000
3,التقريب,24,0,50.581677,0.000000
4,الزّاوية وأنواعها,24,1,26.763095,4.166667
5,الشُّعاعُ والمستقيم,24,0,41.958221,0.000000
6,الضرب في العشرات,24,0,49.177309,0.000000
7,الضرب في العشرات تفاعلي,24,0,47.969522,0.000000
8,الضرب في المئات,24,1,45.950568,4.166667
9,الضرب في المئات ومضاعفاتها,24,0,45.171673,0.000000


In [ ]:
!pip install -q google-generativeai pandas tqdm

In [ ]:
import pandas as pd
import google.generativeai as genai
from tqdm import tqdm
import json
import time
import re

# =========================
# 1. إعداد Gemini API
# =========================

GEMINI_API_KEY = "AQ.Ab8RN6J7Ih_DglGm5eAu0m_MoGVFJATMLvlFPBaRpd7Gdv_9VQ"

genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel("gemini-1.5-flash")

# =========================
# 2. تحميل الملفات
# =========================

video_df = pd.read_csv("/content/video_extracted_skills.csv")
guide_df = pd.read_csv("/content/guide_extracted_skills.csv")

# =========================
# 3. أسماء الأعمدة حسب ملفاتك
# =========================

VIDEO_LESSON_COL = "lesson_title"
VIDEO_SKILL_COL = "skill"
VIDEO_CONCEPT_COL = "concept"
VIDEO_OBJECTIVE_COL = "learning_objective"

GUIDE_LESSON_COL = "lesson_title"
GUIDE_SKILL_COL = "guide_skill"

# =========================
# 4. تنظيف بسيط للنصوص
# =========================

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

video_df[VIDEO_LESSON_COL] = video_df[VIDEO_LESSON_COL].apply(clean_text)
video_df[VIDEO_SKILL_COL] = video_df[VIDEO_SKILL_COL].apply(clean_text)
video_df[VIDEO_CONCEPT_COL] = video_df[VIDEO_CONCEPT_COL].apply(clean_text)
video_df[VIDEO_OBJECTIVE_COL] = video_df[VIDEO_OBJECTIVE_COL].apply(clean_text)

guide_df[GUIDE_LESSON_COL] = guide_df[GUIDE_LESSON_COL].apply(clean_text)
guide_df[GUIDE_SKILL_COL] = guide_df[GUIDE_SKILL_COL].apply(clean_text)

# =========================
# 5. دالة استخراج JSON
# =========================

def extract_json(text):
    text = text.strip()
    text = text.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(text)
    except:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                pass

    return {
        "match_status": "Parsing Error",
        "best_video_skill": "",
        "confidence": 0.0,
        "reason": text
    }

# =========================
# 6. دالة المقارنة بالذكاء الاصطناعي
# =========================

def ai_compare_skill(guide_skill, video_items):
    video_text = "\n\n".join(video_items)

    prompt = f"""
أنت خبير في تحليل مهارات الرياضيات للصف الثالث الأساسي ومقارنتها بدليل المعلم.

المهمة:
قارن مهارة دليل المعلم مع المهارات والمفاهيم والأهداف التعليمية المستخرجة من الفيديو.

مهارة دليل المعلم:
{guide_skill}

المحتوى المستخرج من الفيديو:
{video_text}

صنّف النتيجة إلى واحدة فقط من القيم التالية:

Exact Match
Semantic Match
Covered by General Skill
Covered by Learning Objective
Partial Match
Missing

تعريفات مهمة:
- Exact Match: نفس المهارة تقريبًا مذكورة في الفيديو.
- Semantic Match: المهارة موجودة بمعنى مشابه حتى لو بصياغة مختلفة.
- Covered by General Skill: مهارة الفيديو عامة لكنها تغطي مهارة الدليل التفصيلية.
- Covered by Learning Objective: الهدف التعليمي في الفيديو يغطي مهارة الدليل حتى لو لم تُذكر كمهارة صريحة.
- Partial Match: الفيديو يغطي جزءًا من المهارة فقط.
- Missing: لا توجد تغطية واضحة للمهارة.

مهم جدًا:
إذا كانت مهارة الفيديو عامة مثل "قراءة الأعداد بشكل صحيح"
ومهارة الدليل تفصيلية مثل "قراءة الأعداد التي تحتوي على صفر"
فصنّفها Covered by General Skill.

أرجع JSON فقط بهذا الشكل:

{{
  "match_status": "",
  "best_video_skill": "",
  "confidence": 0.0,
  "reason": ""
}}
"""

    try:
        response = model.generate_content(prompt)
        return extract_json(response.text)
    except Exception as e:
        return {
            "match_status": "API Error",
            "best_video_skill": "",
            "confidence": 0.0,
            "reason": str(e)
        }

# =========================
# 7. تنفيذ المقارنة لكل درس
# =========================

results = []

lessons = guide_df[GUIDE_LESSON_COL].dropna().unique()

for lesson in tqdm(lessons):

    guide_skills = guide_df[
        guide_df[GUIDE_LESSON_COL] == lesson
    ][GUIDE_SKILL_COL].dropna().unique().tolist()

    lesson_video = video_df[
        video_df[VIDEO_LESSON_COL] == lesson
    ]

    video_items = []

    for _, row in lesson_video.iterrows():
        item = f"""
المهارة: {row[VIDEO_SKILL_COL]}
المفهوم: {row[VIDEO_CONCEPT_COL]}
الهدف التعليمي: {row[VIDEO_OBJECTIVE_COL]}
"""
        video_items.append(item)

    for guide_skill in guide_skills:

        if len(video_items) == 0:
            results.append({
                "lesson_title": lesson,
                "guide_skill": guide_skill,
                "best_video_skill": "",
                "match_status": "Missing",
                "confidence": 0.0,
                "reason": "No video skills found for this lesson"
            })
            continue

        ai_result = ai_compare_skill(guide_skill, video_items)

        results.append({
            "lesson_title": lesson,
            "guide_skill": guide_skill,
            "best_video_skill": ai_result.get("best_video_skill", ""),
            "match_status": ai_result.get("match_status", ""),
            "confidence": ai_result.get("confidence", 0.0),
            "reason": ai_result.get("reason", "")
        })

        time.sleep(1)

# =========================
# 8. حفظ نتائج المقارنة
# =========================

results_df = pd.DataFrame(results)

results_df.to_csv(
    "/content/ai_skill_alignment_results.csv",
    index=False,
    encoding="utf-8-sig"
)

results_df.head()

100%|██████████| 51/51 [20:00<00:00, 23.54s/it]


,lesson_title,guide_skill,best_video_skill,match_status,confidence,reason
0,الأعداد ضمن 9999,قراءة العدد بالطُّرق المختلفة āإذا كان āأحد āأ...,,API Error,0.0,400 POST https://generativelanguage.googleapis...
1,الأعداد ضمن 9999,تمثيل العدد على لوحة المنازل,,API Error,0.0,"('Connection aborted.', ConnectionResetError(1..."
2,الأعداد ضمن 9999,تمثيل على المِعداد قد يخطئ الطا,,API Error,0.0,400 POST https://generativelanguage.googleapis...
3,الأعداد ضمن 9999,كتابة الҙأعداد ضمن - āأنْ يتعرّف الطالب مفهوم ...,,API Error,0.0,"('Connection aborted.', ConnectionResetError(1..."
4,الأعداد ضمن 9999,تمثيل العدد على المعداد,,API Error,0.0,404 POST https://generativelanguage.googleapis...


In [ ]:
!pip install -q google-generativeai pandas tqdm

In [ ]:
import pandas as pd
import google.generativeai as genai
from tqdm import tqdm
import json
import time
import re

In [ ]:
GEMINI_API_KEY = "AQ.Ab8RN6LlAM50CeLBGotCQ5jjgL5AqkGHk07D2hY8m5hJM8j9SQ"

genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
video_df = pd.read_csv("/content/video_extracted_skills (1).csv")
guide_df = pd.read_csv("/content/guide_extracted_skills.csv")

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.strip()

    text = re.sub(r"\s+", " ", text)

    return text

In [ ]:
for col in video_df.columns:
    video_df[col] = video_df[col].astype(str).apply(clean_text)

for col in guide_df.columns:
    guide_df[col] = guide_df[col].astype(str).apply(clean_text)

In [ ]:
def extract_json(text):

    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()

    match = re.search(r"\[.*\]", text, re.DOTALL)

    if match:
        text = match.group()

    return json.loads(text)

In [ ]:
def compare_lesson(lesson_name,
                   guide_skills,
                   video_records):

    guide_text = "\n".join([
        f"{i+1}. {s}"
        for i, s in enumerate(guide_skills)
    ])

    video_text = ""

    for i, row in enumerate(video_records):

        video_text += f"""

المهارة {i+1}
المهارة: {row['skill']}
المفهوم: {row['concept']}
الهدف التعليمي: {row['learning_objective']}

"""

    prompt = f"""
أنت خبير مناهج رياضيات.

لدي مهارات من دليل المعلم
ومهارات ومفاهيم وأهداف تعليمية مستخرجة من فيديو شرح الدرس.

اسم الدرس:
{lesson_name}

======================

مهارات دليل المعلم:

{guide_text}

======================

المحتوى المستخرج من الفيديو:

{video_text}

======================

لكل مهارة من دليل المعلم حدد:

Exact Match
Semantic Match
Covered by General Skill
Covered by Learning Objective
Partial Match
Missing

أرجع JSON Array فقط بالشكل التالي:

[
 {{
   "guide_skill":"",
   "best_video_skill":"",
   "match_status":"",
   "confidence":0.95,
   "reason":""
 }}
]

بدون أي نص إضافي.
"""

    response = model.generate_content(prompt)

    return extract_json(response.text)

In [ ]:
results = []

lessons = sorted(
    guide_df["lesson_title"].dropna().unique()
)

In [ ]:
for lesson in tqdm(lessons):

    guide_skills = guide_df[
        guide_df["lesson_title"] == lesson
    ]["guide_skill"].dropna().tolist()

    lesson_video = video_df[
        video_df["lesson_title"] == lesson
    ]

    if len(guide_skills) == 0:
        continue

    if len(lesson_video) == 0:
        continue

    video_records = lesson_video[
        [
            "skill",
            "concept",
            "learning_objective"
        ]
    ].to_dict("records")

    try:

        lesson_results = compare_lesson(
            lesson,
            guide_skills,
            video_records
        )

        for item in lesson_results:

            item["lesson_title"] = lesson

            results.append(item)

        time.sleep(2)

    except Exception as e:

        print("Error:", lesson)

        print(e)

  2%|▏         | 1/51 [03:04<2:33:56, 184.72s/it]ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2227.87ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1571.74ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6931.32ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1570.86ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1318.08ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1573.03ms
  4%|▍         | 2/51 [03:28<1:13:36, 90.13s/it] 

Error: الأعداد ضمن 99999
429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 46.026584465s.


  6%|▌         | 3/51 [03:29<39:19, 49.16s/it]  

Error: التعرف على المتر والسنتيمتر والعلاقة بينهما
429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 45.642719245s.


  8%|▊         | 4/51 [03:29<23:28, 29.98s/it]

...

Error: وحدات قياس الزمن
429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 18.790151514s.


 98%|█████████▊| 50/51 [03:56<00:00,  1.92it/s]

Error: وحدات قياس الطول
429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 18.199377234s.


100%|██████████| 51/51 [03:56<00:00,  4.65s/it]

Error: وحدات قياس الكتلة
429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 17.671164773s.
